In [1]:
from itertools import combinations_with_replacement
from collections import Counter
import math
import pandas as pd


# ------------------------------------------------------------
#  floors 
# ------------------------------------------------------------
def half_floor(x: float) -> float:
    """
    Return the largest half-integer ≤ x.
    Example: 0.74 → 0.5,  2.0 → 2.0,  2.11 → 2.0
    """
    return math.floor(2 * x) / 2

def floor(x: float | int) -> int:
    """
    Return the greatest integer ≤ x.
    """
    n = int(x)           # truncates toward 0
    return n if x >= n else n - 1

# ------------------------------------------------------------
#  partition utilities
# ------------------------------------------------------------
def partitions(n: int) -> list[tuple[int, ...]]:
    """
     Create partitions and order them from longest tuple to smallest tuple
    """
    parts = []
    for r in range(1, n + 1):
        for combo in combinations_with_replacement(range(1, n + 1), r):
            if sum(combo) == n:
                parts.append(combo)
    return sorted(parts, key=lambda t: (-len(t), t))


# ------------------------------------------------------------
#  table construction
# ------------------------------------------------------------
def build_table(N: int) -> pd.DataFrame:
    """
    For a fixed Ksquared = N.
    Only k with 1 ≤ k < N are allowed (enforces Kc1(A) < Ksquared).
    """
    blanks = {str(c): None for c in range(1, 10)}           # columns “1” … “9”
    rows   = []

    for k in range(1, N):                                   # strict < N
        rows.append({                                       
            "Ksquared": N,
            "Kc1(A)":   k,
            "Harder-Narasimhan filtration": None,
            **blanks
        })
        for p in partitions(k):                             # partition rows
            rows.append({
                "Ksquared": N,
                "Kc1(A)":   f"↳ {k}",
                "Harder-Narasimhan filtration": p,
                **blanks
            })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
#  fill the ch₂ columns
# ------------------------------------------------------------
def fill_ch2(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each partition row:

        for each unique m in the tuple with 1 ≤ m ≤ 9
            c1(F₀) = floor(m**2 / Ksquared)
            ch₂    = half_floor(c1(F₀) / 2)
            write ch₂ into column str(m)
    """
    for idx, row in df.iterrows():
        part = row["Harder-Narasimhan filtration"]
        if part is None:                             # skip parent rows
            continue

        N = row["Ksquared"]
        for m in set(part):
            if 1 <= m <= 9:
                c1  = floor(m ** 2 / N)         # squared m per spec
                ch2 = half_floor(c1 / 2)
                df.at[idx, str(m)] = ch2

    return df


# ------------------------------------------------------------
#  add upper-bound column
# ------------------------------------------------------------
def add_upper_bound(df: pd.DataFrame) -> pd.DataFrame:
    """
    ch2(A) upper bound = Σ multiplicity(m) · value_in_column_m
    (Only across columns “1” … “9”; parent rows get None.)
    """
    for idx, row in df.iterrows():
        part = row["Harder-Narasimhan filtration"]
        if part is None:
            df.at[idx, "ch2(A) upper bound"] = None
            continue

        total = 0.0
        for m, mult in Counter(part).items():
            if 1 <= m <= 9:
                col_val = row[str(m)]
                if pd.notna(col_val):
                    total += mult * col_val

        df.at[idx, "ch2(A) upper bound"] = total

    return df

# ------------------------------------------------------------
#  t² lower-bound column
# ------------------------------------------------------------
def add_tsquared_lower_bound(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each partition row compute

        t² = ( ch2(A)                       – 3/2 · Kc1(A)
             + 12 · Kc1(A) / Ksquared )   / ( Ksquared – Kc1(A) )

    using the value already stored in the “ch2(A) upper bound” column.
    Parent rows (where that column is NaN) receive None.
    """
    for idx, row in df.iterrows():
        ch2_upper = row["ch2(A) upper bound"]
        if pd.isna(ch2_upper):                # parent rows → None
            df.at[idx, "tsquared lower bound"] = None
            continue

        ksq  = row["Ksquared"]

        # “Kc1(A)” is an int on parent rows and the string "↳ k" on partitions.
        k_raw = row["Kc1(A)"]
        k     = int(str(k_raw).replace("↳", "").strip())

        denom = ksq - k                       # always > 0 in your construction
        tsq   = (ch2_upper - 1.5 * k + 12 * k / ksq) / denom

        df.at[idx, "tsquared lower bound"] = tsq

    return df

# ------------------------------------------------------------
#  t lower-bound column (square root of t²)
# ------------------------------------------------------------
def add_tlower_bound(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute tlwrbnd = √(t² lower bound) when that value is non‑negative.
    If t² is negative or NaN the entry is left as None.
    """
    for idx, row in df.iterrows():
        tsq = row["tsquared lower bound"]
        if pd.isna(tsq) or tsq < 0:
            df.at[idx, "tlwrbnd"] = 0
        else:
            df.at[idx, "tlwrbnd"] = math.sqrt(tsq)
    return df


# ------------------------------------------------------------
#  demonstration / driver
# ------------------------------------------------------------
if __name__ == "__main__":
    results = {}                                           

    for N in range(1, 10):                                 # 1 ≤ N ≤ 9
        df = build_table(N)
        df = fill_ch2(df)
        add_upper_bound(df)
        df  = add_tsquared_lower_bound(df)
        df  = add_tlower_bound(df)                         
        results[N] = df                                    

        print(f"\n-----  N = {N}  -----")
        print(df.to_string(index=False))


-----  N = 1  -----
Empty DataFrame
Columns: []
Index: []

-----  N = 2  -----
 Ksquared Kc1(A) Harder-Narasimhan filtration    1    2    3    4    5    6    7    8    9 ch2(A) upper bound tsquared lower bound  tlwrbnd
        2      1                         None None None None None None None None None None               None                 None  0.00000
        2    ↳ 1                         (1,)  0.0 None None None None None None None None                0.0                  4.5  2.12132

-----  N = 3  -----
 Ksquared Kc1(A) Harder-Narasimhan filtration    1    2    3    4    5    6    7    8    9 ch2(A) upper bound tsquared lower bound  tlwrbnd
        3      1                         None None None None None None None None None None               None                 None 0.000000
        3    ↳ 1                         (1,)  0.0 None None None None None None None None                0.0                 1.25 1.118034
        3      2                         None None None Non

In [ ]:

import math
import pandas as pd

# 1) helper to add the new column (unchanged)
def add_t2_lower_bound(df: pd.DataFrame) -> pd.DataFrame:
    def _calc(row):
        ch2_upper = row["ch2(A) upper bound"]
        if pd.isna(ch2_upper):
            return None
        ksq = row["Ksquared"]
        k   = int(str(row["Kc1(A)"]).replace("↳", "").strip())
        num   = (2*ksq*ch2_upper) - k * (3 * ksq - 24)
        denom = 2 * ksq * (ksq - k)
        rad   = num / denom
        return math.sqrt(rad) if rad >= 0 else None
    df["t_2 lower bound"] = df.apply(_calc, axis=1)
    return df

# 2) compact view (same as before)
def make_summary(df: pd.DataFrame) -> pd.DataFrame:
    return df[
        ["Ksquared", "Kc1(A)", "ch2(A) upper bound",
         "tsquared lower bound", "tlwrbnd", "t_2 lower bound"]
    ]

# 3) run through every Ksquared block in results
for N in sorted(results):
    if N < 2:      # skip the empty Ksquared = 1 table
        continue
    dfN = results[N].copy()           # don’t clobber the original
    add_t2_lower_bound(dfN)           # ← new column
    print(f"\n=====  Ksquared = {N}  =====")
  
    print(make_summary(dfN).to_string(index=False))